# Load datasets

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler

# Display and visualization defaults
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120

# Importing Libraries

In [2]:
DATA_DIR = Path(r"D:\LSCM - Self Study\Case\scmission\2026\[ROUND 3] CASE\SCMission2026_Round3_Data\SCMission2026_Round3_Data")

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data folder not found: {DATA_DIR}")

# Master and cost tables (IDs are strings so leading zeroes are preserved)
customer_master = pd.read_csv(
    DATA_DIR / "CustomerMaster.csv", dtype={"LocationID": "string"}
)
facility_master = pd.read_csv(
    DATA_DIR / "FacilityMaster.csv", dtype={"LocationID": "string"}
)
product_master = pd.read_csv(
    DATA_DIR / "ProductMaster.csv", dtype={"ProductID": "string"}
)
first_mile_cost = pd.read_csv(
    DATA_DIR / "FirstMileCost.csv",
    dtype={"OriginID": "string", "DestinationID": "string"},
)
last_mile_b2b_cost = pd.read_csv(
    DATA_DIR / "LastMileB2BCost.csv",
    dtype={"OriginID": "string", "DesitnationID": "string"},
).rename(columns={"DesitnationID": "DestinationID"})
transfer_cost = pd.read_csv(
    DATA_DIR / "TransferCost.csv",
    dtype={"OriginID": "string", "DestinationID": "string"},
)
sto = pd.read_csv(
    DATA_DIR / "STO.csv",
    dtype={"OriginID": "string", "DestinationID": "string", "ProductID": "string"},
)

# Excel tables
data_dictionary = pd.read_excel(
    DATA_DIR / "Data_Dictionary.xlsx", sheet_name="Data Dictionary"
)
fc_investment_parameters = pd.read_excel(
    DATA_DIR / "FC_Investment_Parameters.xlsx", sheet_name="FC Investment Parameters"
)

# DailyDemand is about 1 GB: inspect a sample and process the complete file in chunks.
DAILY_DEMAND_DTYPES = {
    "OriginID": "string",
    "DestinationID": "string",
    "ProductID": "string",
    "Type": "category",
}

def read_daily_demand_chunks(chunk_size=250_000):
    return pd.read_csv(
        DATA_DIR / "DailyDemand.csv",
        dtype=DAILY_DEMAND_DTYPES,
        parse_dates=["Date"],
        chunksize=chunk_size,
    )

daily_demand_sample = pd.read_csv(
    DATA_DIR / "DailyDemand.csv",
    dtype=DAILY_DEMAND_DTYPES,
    parse_dates=["Date"],
    nrows=100_000,
)
daily_demand_chunks = read_daily_demand_chunks()

datasets = {
    "customer_master": customer_master,
    "facility_master": facility_master,
    "product_master": product_master,
    "first_mile_cost": first_mile_cost,
    "last_mile_b2b_cost": last_mile_b2b_cost,
    "transfer_cost": transfer_cost,
    "sto": sto,
    "data_dictionary": data_dictionary,
    "fc_investment_parameters": fc_investment_parameters,
    "daily_demand_sample": daily_demand_sample,
}

pd.DataFrame(
    [{"dataset": name, "rows": len(df), "columns": df.shape[1]} for name, df in datasets.items()]
).sort_values("dataset").reset_index(drop=True)

,dataset,rows,columns
0,customer_master,34805,8
1,daily_demand_sample,100000,7
2,data_dictionary,12,4
3,facility_master,55,20
4,fc_investment_parameters,18,15
5,first_mile_cost,36,3
6,last_mile_b2b_cost,38848,3
7,product_master,150,7
8,sto,118579,5
9,transfer_cost,39,3


In [4]:
from pathlib import Path
import pandas as pd

daily_demand_path = Path(
    r"D:\LSCM - Self Study\Case\scmission\2026\[ROUND 3] CASE"
    r"\SCMission2026_Round3_Data\SCMission2026_Round3_Data"
    r"\DailyDemand.csv"
)

daily_demand = pd.read_csv(
    daily_demand_path,
    dtype={
        "OriginID": "string",
        "DestinationID": "string",
        "ProductID": "string",
        "Type": "category",
    },
    parse_dates=["Date"],
)